#### get text content from data
If you use this dataset, please cite the following paper:
Urchs et al. (2025): taz2024full: Analysing German Newspapers for Gender Bias and Discrimination across Decades, Findings of ACL 2025.

format:
- meta data:

    - "published_on": publishing date in the format YYYY-MM-DDThh:mm:ss+01:00

    - "contains_actors": boolean indicating whether person entities were detected

    - "crawled_on": crawling date

    - "language": always "de"

    - "type": always "article"

    - "author": article author

    - "keywords": topic keywords

    - "token_count": number of tokens


- text:

    - "title": article title

    - "teaser": short summary

    - "text": main body of the article

In [1]:
import pandas as pd
import json
from pprint import pprint
import os
import re
import string

import spacy
nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm

def flatten(xss):
    return [x for xs in xss for x in xs]

In [ ]:
c_directory = r"corpus" 

# Iterate over files in directory
dict_year = {}
for name in os.listdir(c_directory):
    dir = os.path.join(c_directory, name)
    for n in os.listdir(dir):
        with open(os.path.join(dir, n)) as f:
            data = json.load(f)
            dict_year[name] = data
print(dict_year.keys())

In [2]:
# separate sentencesy with spacy
def separate_sentences(sents):
    doc = nlp(sents)
    res = []
    for sent in doc.sents:
     res.append(sent.text.lower())
    return res

In [3]:
# keep text content and id only
def save_txt_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in dict_year.keys():
        text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            text_dict[k] = text
            with open(f"{text_dir}_{k}.txt", "w") as f:
                    f.write(json.dumps(separate_sentences(str(text))))

#save_txt_to_file()

#### get sentences with occurences of AI related words
Notes: 
- Englisch and German for certain terms?
- Specific product names
- ML/AI system terminology
- Nouns only
- Abreviations, too



steps:
- build AI vocabulary
- filter by sentence containing at least one AI vocab
- multiply sentences if they contain multiple terms 

In [31]:
# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung']
ki_terms = [x.lower() for x in ki_terms]
regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b


<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_15953/786021808.py:6: SyntaxWarning: invalid escape sequence '\s'
  regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]


#### Sätze säubern, filtern
- Satzzeichen
- (Html, etc)
- POS (use spacy bc TIGER corpus trained)
- Lexikon für Verb- / Adjektivtypen

In [8]:
# preprocessing auxiliary functions

stopwords = nlp.Defaults.stop_words
with open(f"config_files/spacy_stopwords.txt", "w") as f:
        f.write(str(stopwords))

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus.lower().split() if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    try:
        out = [token for token in token_list if not token.is_stop]
        return out
    except:
        return None

def determine_negation(doc):
    negation_tokens = [tok for tok in doc if tok.dep_ == 'neg']
    negation_head_tokens = [token.head for token in negation_tokens]
    for token in negation_head_tokens:
        return [child for child in token.children]


##### get sentences with ki_terms

In [ ]:
# filter ki_terms
c_txt_dir = r"corpus_texts" 
ki_sentences = {}
for dir in os.listdir(c_txt_dir):
    res_tmp = []
    with open(os.path.join(c_txt_dir,dir)) as f:
        sent_l = json.load(f)
        # could also account for lemmata, etc
    sent_l = [remove_punctuation(sent) for sent in sent_l]
    #if any([re.search(regex_ki, sent) != None for sent in sent_l]):
    for sent in sent_l:
        if re.search(regex_ki, sent):
           #print("res",sent, re.search(regex_ki, sent).string)
            res_tmp.append(sent)
        # add to dict
        if len(res_tmp) > 0:
            ki_sentences [dir] =  res_tmp

# save to file
filtered_sentences_json = json.dumps(ki_sentences)

pprint(len(filtered_sentences_json))
#pprint(filtered_sentences_json)

230779
('{"_5569995.txt": ["die welt und ich 2000 ki"], "_5317962.txt": ["die welt '
 'und ich 2000 ki"], "_5573067.txt": ["die welt und ich 2000 ki"], '
 '"_721946.txt": ["tip h hb hh ki berlin calling deutschland", "filmdienst bhv '
 'del h hb hh hl ki ol bride wars  beste feindinnen", "bhv del h hb hh hl ki '
 'ol buddenbrooks deutschland 2008 r dr heinrich breloer d armin muellerstahl '
 'iris berben \\u201e\\u00fcber drei generationen hinweg schildert thomas mann '
 'in seinem roman den aufstieg und fall einer l\\u00fcbecker kaufmannsfamilie '
 'im 19 jahrhundert", "keine gro\\u00dfen gef\\u00fchle sondern verhaltene '
 'trauer  und am ende ein gef\\u00fchl der wehmut das die bilder lange '
 'nachwirken l\\u00e4sst\\u201c cinema h hb hh hl ki ol chiko deutschland 2007 '
 'r \\u00f6zg\\u00fcr yildirim d denis moschitto moritz bleibtreu '
 '\\u201echiko\\u2018 ein t\\u00fcrkischer kleinganove und sein kumpel tibet '
 'tr\\u00e4umen davon eines tages im wei\\u00dfen mercedes \\u00fcb

In [40]:
#save dict to file
with open("generated_data/KI_sentences.json", "w") as f:
        f.write(str(filtered_sentences_json))


### Daten veranschaulichen:
- Häufigkeit der KI Terme
- Häufigste Wörter
- length of sentences / number of clauses
- number / type of verbs
- number / type of adjektives

In [ ]:
plots_dir = "plots/"